## 2.2 编程题

In [7]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表，生成n-gram特征和标签。
    
    参数:
        text (str): 输入文本
        n (int): 滑动窗口长度（特征词数）
    
    返回:
        vocab (dict): 词到整数ID的映射（按频率降序排列）
        features (list of list of str): 特征窗口（词列表）
        labels (list of str): 每个窗口对应的下一个词（若有）
    """
    # 1. 转小写，去除非字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留小写字母和空格
    
    # 2. 分词
    tokens = text.split()
    
    # 3. 构建词汇表（按频率降序）
    freq = Counter(tokens)
    # 按频率降序，频率相同则按字母顺序（可选）
    sorted_words = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(tokens) - n):  # 保证有后续词
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n])
    
    return vocab, features, labels

text = "The quick brown fox jumps over the lazy dog. The dog sleeps."
n = 3

vocab, features, labels = preprocess_text(text, n)

print("词汇表 (按频率排序):", vocab)
print("\n特征 (前5个):", features[:5])
print("标签 (前5个):", labels[:5])
print("\n总样本数:", len(features))

词汇表 (按频率排序): {'the': 0, 'dog': 1, 'brown': 2, 'fox': 3, 'jumps': 4, 'lazy': 5, 'over': 6, 'quick': 7, 'sleeps': 8}

特征 (前5个): [['the', 'quick', 'brown'], ['quick', 'brown', 'fox'], ['brown', 'fox', 'jumps'], ['fox', 'jumps', 'over'], ['jumps', 'over', 'the']]
标签 (前5个): ['fox', 'jumps', 'over', 'the', 'lazy']

总样本数: 9


## 3.2 编程题（RNN单元前向与反向）

In [2]:
import numpy as np

def rnn_step_forward(x_t, prev_h, W_hx, W_hh, b_h):
    """
    RNN单元前向传播。
    参数:
        x_t: (batch_size, input_size)
        prev_h: (batch_size, hidden_size)
        W_hx: (input_size, hidden_size)
        W_hh: (hidden_size, hidden_size)
        b_h: (hidden_size,)
    返回:
        h_t: (batch_size, hidden_size)
        缓存: (x_t, prev_h, W_hx, W_hh, b_h, h_t) 用于反向
    """
    h_t = np.tanh(x_t @ W_hx + prev_h @ W_hh + b_h)
    cache = (x_t, prev_h, W_hx, W_hh, b_h, h_t)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    """
    RNN单元反向传播。
    参数:
        dh_next: (batch_size, hidden_size) 损失对h_t的梯度
        cache: 前向缓存
    返回:
        dx_t: (batch_size, input_size)
        dh_prev: (batch_size, hidden_size)
        dW_hx: (input_size, hidden_size)
        dW_hh: (hidden_size, hidden_size)
        db_h: (hidden_size,)
    """
    x_t, prev_h, W_hx, W_hh, b_h, h_t = cache
    # tanh导数: 1 - h_t^2
    dtanh = dh_next * (1 - h_t ** 2)  # element-wise
    
    # 计算各梯度
    dx_t = dtanh @ W_hx.T
    dh_prev = dtanh @ W_hh.T
    dW_hx = x_t.T @ dtanh
    dW_hh = prev_h.T @ dtanh
    db_h = np.sum(dtanh, axis=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

## 4.2 编程题（双向RNN编码器）

In [3]:
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, hidden_dim, num_layers=1):
    """
    双向RNN编码器，返回每个时间步拼接状态和最终拼接状态。
    参数:
        X: (seq_len, batch, input_dim)
        hidden_dim: 每个方向的隐藏单元数
        num_layers: RNN层数（默认1）
    返回:
        outputs: (seq_len, batch, 2*hidden_dim) 每个时间步前后向拼接
        final_state: (batch, 2*hidden_dim) 最终时间步的前后向拼接（作为序列表示）
    """
    # 使用PyTorch的RNN
    rnn = nn.RNN(
        input_size=X.size(2),
        hidden_size=hidden_dim,
        num_layers=num_layers,
        bidirectional=True,
        batch_first=False  # 保持 (seq, batch, feature)
    )
    # 初始化隐藏状态 (num_layers*2, batch, hidden_dim)
    h0 = torch.zeros(num_layers * 2, X.size(1), hidden_dim)
    outputs, h_n = rnn(X, h0)
    # outputs: (seq_len, batch, num_directions * hidden_dim) = (seq_len, batch, 2*hidden_dim)
    # h_n: (num_layers*2, batch, hidden_dim)
    
    # 最终时间步的拼接状态：取最后前向和最后后向
    # 对于单层，h_n[0] 是前向最后，h_n[1] 是后向最后
    if num_layers == 1:
        final_forward = h_n[0]   # (batch, hidden_dim)
        final_backward = h_n[1]  # (batch, hidden_dim)
    else:
        # 多层：取最后一层的前后向
        final_forward = h_n[2*(num_layers-1)]   # 最后一层前向
        final_backward = h_n[2*(num_layers-1)+1] # 最后一层后向
    final_state = torch.cat([final_forward, final_backward], dim=1)  # (batch, 2*hidden_dim)
    
    return outputs, final_state



## 5.2 编程题（CBOW 完整 softmax）

In [4]:
import numpy as np

def cbow_forward(context_indices, target_idx, W, W_out):
    """
    CBOW模型前向传播和损失计算（完整softmax）。
    参数:
        context_indices: (batch_size, context_size) 每个样本的上下文词索引
        target_idx: (batch_size,) 每个样本的中心词索引
        W: (vocab_size, embed_dim) 输入嵌入矩阵
        W_out: (embed_dim, vocab_size) 输出权重矩阵
    返回:
        loss: 标量，平均交叉熵损失
    """
    batch_size, context_size = context_indices.shape
    vocab_size, embed_dim = W.shape
    
    # 获取上下文嵌入 (batch, context_size, embed_dim)
    emb = W[context_indices]  # (batch, context_size, embed_dim)
    # 平均上下文向量
    h = np.mean(emb, axis=1)  # (batch, embed_dim)
    
    # 计算得分 (logits)
    logits = h @ W_out  # (batch, vocab_size)
    
    # softmax 概率
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))  # 数值稳定
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    # 交叉熵损失： -log(probs[target])
    target_probs = probs[np.arange(batch_size), target_idx]
    loss = -np.mean(np.log(target_probs + 1e-8))  # 加小常数避免log(0)
    
    return loss

# 示例测试（小规模）
V, d = 5, 3
context_size = 2
batch = 2
W = np.random.randn(V, d)
W_out = np.random.randn(d, V)
context = np.random.randint(0, V, size=(batch, context_size))
target = np.random.randint(0, V, size=(batch,))
loss = cbow_forward(context, target, W, W_out)
print("CBOW损失:", loss)

CBOW损失: 2.3053686154069037


## 6.2 编程题（多头注意力）

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层（不区分Q/K/V，可分别定义）
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        返回: (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 线性投影并分头
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 重塑为 (seq_len, batch, num_heads, d_k) 并交换维度为 (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(0, 1)  # (batch, num_heads, seq_len, d_k)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(0, 1)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(0, 1)
        
        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (batch, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_k)
        
        # 合并头维度
        attn_output = attn_output.transpose(0, 1).transpose(1, 2).contiguous()  # (seq_len, batch, num_heads, d_k)
        attn_output = attn_output.view(seq_len, batch, self.d_model)  # (seq_len, batch, d_model)
        
        # 最终线性层
        output = self.W_o(attn_output)
        return output

# 测试
d_model = 4
num_heads = 2
seq_len, batch = 3, 2
X = torch.randn(seq_len, batch, d_model)
mha = MultiHeadAttention(d_model, num_heads)
out = mha(X)
print(out.shape)  

torch.Size([3, 2, 4])
